In [ ]:
import pandas as pd
from collections import defaultdict

FILE_PATH = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026.xlsx"
MAIN_SHEET = "Master Data "
PPM_SHEET = "Part Production Master "

ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]

def main():
    xls = pd.ExcelFile(FILE_PATH)
    
    # Load main data
    df = pd.read_excel(xls, MAIN_SHEET)
    df.columns = df.columns.str.strip()
    
    # Optional: load cycle time override from PPM (Machine column = cycle time)
    try:
        df_cycle = pd.read_excel(xls, PPM_SHEET)
        df_cycle.columns = df_cycle.columns.str.strip()
        cycle_map = dict(zip(
            df_cycle["Material"].astype(str).str.strip(),
            pd.to_numeric(df_cycle["Machine"], errors='coerce')
        ))
        df["Cycle Time"] = df["Child Part"].astype(str).str.strip().map(cycle_map).fillna(df["Cycle Time"])
    except:
        print("Note: Could not load cycle time from Part Production Master. Using main sheet values.")
    
    # Filter rows with positive Daily Plan
    df = df[df["Daily Plan"].fillna(0) > 0].copy()
    
    # Quantity to produce
    df["qty"] = (df["Plan"] - df["Inventory_25"].fillna(0)).clip(lower=0)
    
    # Only rows needing production
    df_need = df[df["qty"] > 0].copy()
    
    if df_need.empty:
        print("\nNo parts need production (all covered by Inventory_25).")
        return
    
    # Load tracking (in hours)
    machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
    plan_rows = []
    
    for _, row in df_need.iterrows():
        child = str(row["Child Part"]).strip()
        vm_str = str(row.get("Vertical Machines", ""))
        possible_machines = [m.strip() for m in vm_str.split(",") if m.strip() in ALLOWED_MACHINES]
        
        if not possible_machines:
            continue
        
        # Choose machine with least current load
        machine = min(possible_machines, key=lambda m: machine_load[m])
        
        # Time in hours (Cycle Time assumed in seconds)
        time_hours = (row["qty"] * row["Cycle Time"]) / 3600.0
        
        machine_load[machine] += time_hours
        
        plan_rows.append({
            "Child Part": child,
            "Switch Part": row["Switch Part Number"],
            "Machine": machine,
            "Quantity": int(round(row["qty"])),
            "Time (hours)": round(time_hours, 2)
        })
    
    if not plan_rows:
        print("\nNo valid production assignments (no eligible machines found in Vertical Machines).")
        return
    
    df_plan = pd.DataFrame(plan_rows)
    
    # Sort plan by machine and then by time descending
    df_plan = df_plan.sort_values(["Machine", "Time (hours)"], ascending=[True, False])
    
    # Load summary
    df_load = pd.DataFrame({
        "Machine": list(machine_load.keys()),
        "Total Load (hours)": [round(v, 2) for v in machine_load.values()]
    }).sort_values("Machine")
    
    # Display directly
    print("\n" + "="*70)
    print("          PRODUCTION PLAN (per row - independent)")
    print("="*70)
    print(df_plan.to_string(index=False))
    
    print("\n" + "="*70)
    print("          MACHINE LOAD SUMMARY")
    print("="*70)
    print(df_load.to_string(index=False))
    
    print("\nDone. No Excel file created.")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# Load sheets
# =============================
master = pd.read_excel("input.xlsx", sheet_name="Master Data")
ppm = pd.read_excel("input.xlsx", sheet_name="Part Production Master")

# =============================
# Clean numeric columns
# =============================
num_cols = [
    "Sub Count", "Inventory_25", "Daily Plan",
    "Monthly Requirement 1", "Minimum Quantity", "Plan"
]
for c in num_cols:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

# =============================
# Build Cycle Time Lookup
# =============================
ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce")
cycle_time_map = dict(zip(ppm["Material"], ppm["Machine"]))

# =============================
# Constants
# =============================
ALLOWED_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}
MACHINE_CAPACITY = 3960

# =============================
# Helpers
# =============================
def normalize_machine(m):
    if not m:
        return None
    m = m.upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# Tracking
# =============================
machine_load = {m: 0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejections = []

# =============================
# Compute mandatory load → target
# =============================
master["Net Required Qty"] = master["Plan"] - master["Inventory_25"]

mandatory_load = 0
for _, r in master.iterrows():
    if r["Daily Plan"] > 0 and r["Net Required Qty"] > 0:
        ct = cycle_time_map.get(r["Child Part"], 0)
        mandatory_load += r["Net Required Qty"] * ct

TARGET_LOAD = mandatory_load / len(ALLOWED_MACHINES) if mandatory_load else 0

# =============================
# MAIN LOOP (ROW-WISE AS YOU DEFINED)
# =============================
for _, row in master.iterrows():

    # ---- DAILY PLAN GATE ----
    if row["Daily Plan"] <= 0:
        continue

    child = row["Child Part"]
    inventory = row["Inventory_25"]
    plan_qty = row["Plan"]
    net_qty = plan_qty - inventory

    cycle_time = cycle_time_map.get(child, 0)
    if cycle_time <= 0:
        continue

    # ---- MACHINE PARSING ----
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)
    vertical = [normalize_machine(t) for t in tokens if normalize_machine(t)]
    eligible = [m for m in vertical if m in ALLOWED_MACHINES]

    if not eligible:
        if net_qty > 0:
            rejections.append((child, "No eligible target machine"))
        continue

    # ---- Decide quantity to produce ----
    if net_qty > 0:
        qty_to_make = net_qty                # mandatory
    else:
        qty_to_make = abs(net_qty)           # balancing-only upper bound

    remaining_time = qty_to_make * cycle_time

    def score(m, alloc):
        return abs((machine_load[m] + alloc) - TARGET_LOAD)

    eligible.sort(key=lambda m: score(
        m, min(MACHINE_CAPACITY - machine_load[m], remaining_time)
    ))

    for m in eligible:
        if remaining_time <= 0:
            break

        # Balancing-only restriction
        if net_qty <= 0 and machine_load[m] >= TARGET_LOAD:
            continue

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc = min(available, remaining_time)
        qty = alloc / cycle_time

        machine_load[m] += alloc
        remaining_time -= alloc

        machine_plan[m].append({
            "Child Part": child,
            "Switch Part": row["Switch Part Number"],
            "Quantity": round(qty, 2),
            "Time (min)": round(alloc, 2)
        })

    if net_qty > 0 and remaining_time > 0:
        rejections.append((child, "Capacity shortfall"))

# =============================
# DISPLAY OUTPUT
# =============================
print("\n========== MACHINE-WISE PLAN ==========\n")
for m in sorted(ALLOWED_MACHINES):
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in sorted(ALLOWED_MACHINES)
]))

print("\n========== REJECTIONS ==========\n")
if rejections:
    display(pd.DataFrame(rejections, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")
